In [1]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 13.5 MB/s eta 0:00:00


In [2]:
import json
import random
from collections import defaultdict

print("=" * 55)
print("📊 做市商 Day 10 — Volume Profile（成交量分布）")
print("=" * 55)

# ===== 第一步：生成模拟成交数据 =====
# 假设过去24h有200笔成交
# 价格围绕$62,768波动，大部分成交集中在$62,500-$63,000

random.seed(42)
trades = []
current_price = 62768

for i in range(200):
    # 让大部分成交在当前价格附近，少数远离
    noise = random.gauss(0, 120)  # 正态分布，标准差$120
    price = round(current_price + noise, 2)
    volume = round(random.uniform(0.1, 2.0), 4)
    trades.append((price, volume))

print(f"\n模拟过去24h成交：{len(trades)} 笔")
print(f"价格范围：${min(t[0] for t in trades):.0f} ~ ${max(t[0] for t in trades):.0f}")

# ===== 第二步：按价格分组统计成交量 =====
# 按每$10一个档位分组

volume_profile = defaultdict(float)

for price, vol in trades:
    bucket = round(price / 10) * 10  # 归到最近的$10档
    volume_profile[bucket] += vol

# ===== 第三步：找 Point of Control (POC) =====
# 成交量最大的价格档位

poc_price = max(volume_profile, key=volume_profile.get)
poc_volume = volume_profile[poc_price]

print(f"\n📌 Point of Control (POC/成交量最大点)：")
print(f"   价格：${poc_price:.0f}")
print(f"   成交量：{poc_volume:.2f} BTC")

# ===== 第四步：计算价值区间 (Value Area) =====
# 从POC往两边扩展，累加成交量到总成交量的70%

total_vol = sum(volume_profile.values())
target_vol = total_vol * 0.70  # 价值区间包含70%的成交量

# 按价格排序
sorted_prices = sorted(volume_profile.keys())
poc_index = sorted_prices.index(poc_price)

# 从POC开始，往两边扩展
accumulated = volume_profile[poc_price]
lower_idx = poc_index
upper_idx = poc_index

while accumulated < target_vol:
    # 尝试往左扩一个
    if lower_idx > 0:
        lower_idx -= 1
        accumulated += volume_profile[sorted_prices[lower_idx]]
    # 尝试往右扩一个
    if upper_idx < len(sorted_prices) - 1 and accumulated < target_vol:
        upper_idx += 1
        accumulated += volume_profile[sorted_prices[upper_idx]]
    # 如果左右都到边界了，退出
    if lower_idx == 0 and upper_idx == len(sorted_prices) - 1:
        break

va_low = sorted_prices[lower_idx]
va_high = sorted_prices[upper_idx]
va_percent = (accumulated / total_vol) * 100

print(f"\n📊 价值区间 (Value Area / VA)：")
print(f"   下边界：${va_low:.0f}")
print(f"   上边界：${va_high:.0f}")
print(f"   区间宽度：${va_high - va_low:.0f}")
print(f"   包含成交量：{accumulated:.2f} BTC ({va_percent:.1f}%)")

# ===== 第五步：判断当前价格在什么位置 =====

current = current_price
print(f"\n🔍 当前价格：${current}")

if current < va_low:
    print(f"   → 当前价格在价值区间下方 ❄️ 折价区")
    print(f"   → 信号：价格可能反弹(买入区域)")
elif current > va_high:
    print(f"   → 当前价格在价值区间上方 🔥 溢价区")
    print(f"   → 信号：价格可能回落(卖出区域)")
else:
    print(f"   → 当前价格在价值区间内 ✅ 正常范围")
    print(f"   → 信号：没有明显的回归压力")

# ===== 第六步：跟之前学的工具一起用 =====

print(f"\n{'=' * 55}")
print("📋 综合判断（Volume Profile + 订单簿不平衡）")
print("=" * 55)

# 模拟一个场景
# 假设：订单簿不平衡显示买方占优 +30%
# Volume Profile显示当前价格在价值区间内

orderbook_imbalance = 0.30   # 模拟的订单簿不平衡度
price_in_va = (va_low <= current <= va_high)

print(f"\n   订单簿不平衡度：+{orderbook_imbalance*100:.0f}% 买方占优 🟢")
print(f"   Volume Profile：价格在价值区间{'内 ✅' if price_in_va else '外 ⚠️'}")

if orderbook_imbalance > 0.15 and price_in_va:
    print(f"\n   🟢 综合信号：买入信号加强！")
    print(f"   买方挂单多 + 价格在合理区间 = 上涨动力充足")
elif orderbook_imbalance < -0.15 and not price_in_va:
    print(f"\n   🔴 综合信号：卖出信号加强！")
    print(f"   卖方挂单多 + 价格在溢价区 = 回调风险大")
elif orderbook_imbalance > 0.15 and not price_in_va:
    print(f"\n   🟡 综合信号：矛盾信号")
    print(f"   买方挂单多但价格已经跑出价值区间 = 小心追高")
else:
    print(f"\n   ⚪ 综合信号：中性")
    print(f"   多个指标没有形成一致信号 = 观望")

print(f"\n{'=' * 55}")
print("📌 小结")
print("=" * 55)
print("""
Volume Profile 告诉你"大家都在哪个价位成交"
订单簿不平衡 告诉你"现在谁在挂单"

两个一起看：
- Volume Profile看"公平价格"在哪里
- 订单簿看"现在买卖力量"谁强

如果订单簿强 + 价格在公平价格附近 = 方向明确
如果订单簿强 + 价格跑远了 = 可能有陷阱
""")

📊 做市商 Day 10 — Volume Profile（成交量分布）

模拟过去24h成交：200 笔
价格范围：$62454 ~ $63048

📌 Point of Control (POC/成交量最大点)：
   价格：$62800
   成交量：10.90 BTC

📊 价值区间 (Value Area / VA)：
   下边界：$62680
   上边界：$62910
   区间宽度：$230
   包含成交量：150.49 BTC (71.7%)

🔍 当前价格：$62768
   → 当前价格在价值区间内 ✅ 正常范围
   → 信号：没有明显的回归压力

📋 综合判断（Volume Profile + 订单簿不平衡）

   订单簿不平衡度：+30% 买方占优 🟢
   Volume Profile：价格在价值区间内 ✅

   🟢 综合信号：买入信号加强！
   买方挂单多 + 价格在合理区间 = 上涨动力充足

📌 小结

Volume Profile 告诉你"大家都在哪个价位成交"
订单簿不平衡 告诉你"现在谁在挂单"

两个一起看：
- Volume Profile看"公平价格"在哪里
- 订单簿看"现在买卖力量"谁强

如果订单簿强 + 价格在公平价格附近 = 方向明确
如果订单簿强 + 价格跑远了 = 可能有陷阱

